# 01 — Bronze Transactions Ingestion

**Purpose:** Incrementally ingest daily e-commerce CSV files into a raw Delta Lake Bronze table.

**Flow:** `Daily CSV files → Auto Loader → Structured Streaming → Bronze Delta`

The Bronze layer keeps source events close to their original form while adding technical lineage metadata. Auto Loader discovers new files, `schemaLocation` stores inferred schema metadata, and the streaming checkpoint tracks ingestion progress across runs.


In [ ]:
from pyspark.sql.functions import col, current_timestamp

SOURCE_PATH = "/Volumes/ecommerce_lakehouse/raw/source_files/"
SCHEMA_PATH = "/Volumes/ecommerce_lakehouse/raw/pipeline_metadata/transactions_schema"
BRONZE_CHECKPOINT = "/Volumes/ecommerce_lakehouse/raw/pipeline_metadata/transactions_checkpoint"

BRONZE_TABLE = "ecommerce_lakehouse.bronze.transactions_raw"


## Read new source files with Auto Loader

`cloudFiles` enables incremental file discovery. `availableNow` is applied at the sink later so each scheduled run processes all currently available unprocessed files and then stops.


In [ ]:
transactions_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.schemaLocation", SCHEMA_PATH)
        .option("cloudFiles.inferColumnTypes", "true")
        .load(SOURCE_PATH)
)


## Add ingestion lineage

Two technical columns are added without changing the business meaning of the source events:

- `_ingested_at` — when the row entered Bronze.
- `_source_file` — source file path used for lineage and troubleshooting.

Auto Loader's `_rescued_data` column is retained so unexpected schema values are not silently discarded.


In [ ]:
bronze_df = (
    transactions_stream
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source_file", col("_metadata.file_path"))
)


## Persist the Bronze stream to Delta Lake

The checkpoint makes repeated job runs incremental: previously processed files are not ingested again. Delta provides transactional writes for the streaming sink.


In [ ]:
bronze_query = (
    bronze_df.writeStream
        .format("delta")
        .option("checkpointLocation", BRONZE_CHECKPOINT)
        .trigger(availableNow=True)
        .toTable(BRONZE_TABLE)
)

bronze_query.awaitTermination()
